# 02 — Fine-Tuning

This notebook fine-tunes `distilbert-base-uncased` on the SST-2 binary sentiment classification task using the Hugging Face `Trainer` API.

> **Note:** This notebook was trained on Google Colab (T4 GPU) due to MPS performance limitations on Apple Silicon for full training loops. Training time: ~15-20 minutes on T4.

## What this notebook covers
- Loading and tokenizing the full SST-2 training set (67,349 examples)
- Fine-tuning DistilBERT with a randomly initialized classification head
- Monitoring validation accuracy and loss across epochs
- Saving the best checkpoint based on validation loss

## Training configuration
| Parameter | Value |
|---|---|
| Base model | `distilbert-base-uncased` |
| Epochs | 3 |
| Batch size (train) | 32 |
| Batch size (eval) | 64 |
| Learning rate | 2e-5 |
| Max sequence length | 128 |

## Results per epoch
| Epoch | Training Loss | Validation Loss | Accuracy |
|---|---|---|---|
| 1 | 0.1797 | 0.2720 | 90.37% |
| 2 | 0.1182 | 0.3091 | 90.02% |
| 3 | 0.0803 | 0.3718 | 90.25% |

## Interpretation
Training loss decreases steadily across epochs while validation loss increases — a classic sign of mild overfitting. Validation accuracy remains stable (~90%), confirming that the model generalizes well. `load_best_model_at_end=True` ensures the saved model corresponds to **epoch 1**, which achieved the lowest validation loss.

The fine-tuned model is saved to `models/distilbert-sst2-finetuned/` for evaluation in `03_evaluation.ipynb`.

In [1]:
!pip install transformers datasets evaluate accelerate scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


In [2]:
# Blocco 1 — Imports e setup

import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset
import evaluate

# Device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

Device: cuda
PyTorch: 2.10.0+cu128


In [3]:
# Blocco 2 — Caricamento e tokenizzazione dataset

# Caricamento
dataset = load_dataset("glue", "sst2")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Funzione di tokenizzazione
def tokenize_function(examples):
    return tokenizer(
        examples["sentence"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Tokenizzazione dell'intero dataset
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

print("Dataset tokenizzato:")
print(tokenized_dataset)
print("\nColonne dopo tokenizzazione:")
print(tokenized_dataset["train"].column_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Dataset tokenizzato:
DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1821
    })
})

Colonne dopo tokenizzazione:
['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask']


In [4]:
# Blocco 3 — Preparazione dataset per il Trainer

# Rimuoviamo le colonne non necessarie
tokenized_dataset = tokenized_dataset.remove_columns(
    ["sentence", "idx"]
)

# Rinominiamo 'label' in 'labels' — formato atteso dal Trainer
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

# Impostiamo il formato PyTorch
tokenized_dataset.set_format("torch")

print("Colonne finali:", tokenized_dataset["train"].column_names)
print("Esempio singolo dal training set:")
print(tokenized_dataset["train"][0])

Colonne finali: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
Esempio singolo dal training set:
{'labels': tensor(0), 'input_ids': tensor([  101,  5342,  2047,  3595,  8496,  2013,  1996, 18643,  3197,   102,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     

In [5]:
# Blocco 4 — Modello e metrica

# Modello
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)
model = model.to(device)

# Metrica
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

print("Modello e metrica pronti.")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modello e metrica pronti.


In [6]:
# Blocco 5 — TrainingArguments

training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    load_best_model_at_end=True,
)

print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=Fals

In [7]:
# Blocco 6 — Trainer e avvio training

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.179691,0.272017,0.903670
2,0.118227,0.309117,0.900229
3,0.080340,0.371857,0.902523


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=6315, training_loss=0.14129280972273126, metrics={'train_runtime': 2113.7927, 'train_samples_per_second': 95.585, 'train_steps_per_second': 2.988, 'total_flos': 6691160124062208.0, 'train_loss': 0.14129280972273126, 'epoch': 3.0})

In [9]:
trainer.save_model("./distilbert-sst2-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
from google.colab import files
import shutil
shutil.make_archive("distilbert-sst2-finetuned", "zip", "./distilbert-sst2-finetuned")
files.download("distilbert-sst2-finetuned.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>